# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Print metadata summary
meta = dataset.metadata
print(f"Title: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"License: {meta.license}\n")
print(f"Published: {meta.datePublished}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Let's examine what record sets and fields are defined in the dataset's Croissant schema.

In [ ]:
# List all record sets and associated field ids in the dataset
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the dataset schema.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}")
        if 'field' in rs:
            print("  Fields:")
            fields = rs['field']
            # field can be a list or a dict
            if isinstance(fields, dict):
                fields = [fields]
            for f in fields:
                fid = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
                print(f"    - {fid}")
        else:
            print("  (This record set does not define explicit fields)")

### Show a sample record set
If the dataset provides downloadable data, you can list a few data rows by referencing the record set's `@id`.

In [ ]:
# Preview records for the first record set (if available)
if not record_sets:
    print("No record sets to show records from.")
else:
    record_set_id = record_sets[0]['@id']
    print(f"Sample records for record set '@id': {record_set_id}")
    count = 0
    for rec in dataset.records(record_set=record_set_id):
        print(rec)
        count += 1
        if count >= 3:
            break

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis.

The following uses `@id` from the overview for extraction. Adjust to your chosen record set(s) as needed.

In [ ]:
# Prepare to load all record sets into pandas DataFrames
dataframes = {}
rs_ids = []
if not record_sets:
    print("No record sets available for extraction.")
else:
    for rs in record_sets:
        rs_id = rs['@id']
        rs_ids.append(rs_id)
        print(f"Loading records for RecordSet: {rs_id}")
        rows = list(dataset.records(record_set=rs_id))
        if rows:
            df = pd.DataFrame(rows)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame with columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No records found for RecordSet: {rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Adjust the IDs below to match those of the numeric field(s) and grouping field(s) from your record set.

In [ ]:
# Example: Select an available numeric and group field for analysis
if not dataframes:
    print("No DataFrames available for EDA.")
else:
    first_rs_id = rs_ids[0]
    df = dataframes[first_rs_id]
    # Choose fields by column name - you may need to adjust these based on the actual DataFrame
    print("Available columns:", df.columns.tolist())

    # For demonstration, select the first numeric column
    numeric_field_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
    else:
        # fallback: pick the first column
        numeric_field = df.columns[0]
    print(f"Using '{numeric_field}' as numeric field for analysis.")

    threshold = 10  # Example threshold for demonstration
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try groupby on the first non-numeric field if available
    group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"Grouping by field '{group_field}'")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Adjust field IDs appropriately.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If prior analysis produced results
if not dataframes or not numeric_field_candidates:
    print("No numeric data available for visualization.")
else:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping field available, boxplot
    if group_field_candidates:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded dataset metadata and inspected the available record sets and fields (by `@id`).
- Extracted records into DataFrames and demonstrated basic filtering and normalization of numeric fields.
- Visualized data distributions with histograms and boxplots where possible.

For further analysis, refer to the dataset documentation for field meanings, and adjust code above to focus on fields of specific scientific interest.